# BÀI TẬP 3.1: HỌC KHÔNG GIÁM SÁT (Máy học – Ts. Nguyễn An Tế)
1. Cài đặt các độ đo: Silhouette, Calinski-Harabasz, Davies-Bouldin, Dunn, DBCV · 2. k-Means và Fuzzy C-Means tự cài đặt trên `Shopping.csv` · 3. OOP

> Đặt `Shopping.csv` trong `./Data`. Nếu không có file, notebook tự sinh dữ liệu khách hàng mô phỏng (Age, Annual Income, Spending Score) để code vẫn chạy.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.preprocessing import StandardScaler
from sklearn import metrics as skm          # chỉ dùng để ĐỐI CHIẾU kết quả tự cài đặt
warnings.filterwarnings('ignore'); np.random.seed(0)
folder = './Data/'; os.makedirs(folder, exist_ok=True)

---
## 1. Cài đặt các độ đo đánh giá gom cụm (không dùng sklearn)
| Độ đo | Ý nghĩa | Tốt khi | KHI NÀO DÙNG |
|---|---|---|---|
| Silhouette | (b−a)/max(a,b) ∈ [−1,1] | **lớn** | Chọn k; cụm lồi, so sánh trực quan |
| Calinski-Harabasz | tỷ số phân tán giữa cụm / trong cụm | **lớn** | Cụm lồi, gọn (k-Means); tính nhanh |
| Davies-Bouldin | trung bình độ tương tự giữa cụm gần nhất | **nhỏ** | Cụm lồi, so sánh các k |
| Dunn | khoảng cách tối thiểu giữa cụm / đường kính cụm lớn nhất | **lớn** | Cụm tách bạch; nhạy với outlier |
| DBCV | độ đo dựa trên **mật độ** (MST + mutual reachability) | **lớn** (∈[−1,1]) | Cụm **hình dạng bất kỳ**, có nhiễu (DBSCAN/HDBSCAN) – các độ đo trên không phù hợp |

In [ ]:
class Cluster_Validity:
    """Các độ đo gom cụm tự cài đặt. X: (n, d); labels: (n,). Nhãn -1 = nhiễu (chỉ DBCV xử lý)."""

    @staticmethod
    def silhouette(X, labels):
        X = np.asarray(X, float); labels = np.asarray(labels); D = cdist(X, X); ks = np.unique(labels)
        if len(ks) < 2: return 0.0
        s = np.zeros(len(X))
        for i in range(len(X)):
            same = labels == labels[i]; n_same = same.sum()
            if n_same == 1: continue                               # cụm 1 điểm: s = 0
            a = D[i, same].sum() / (n_same - 1)
            b = min(D[i, labels == k].mean() for k in ks if k != labels[i])
            s[i] = (b - a) / max(a, b)
        return s.mean()

    @staticmethod
    def calinski_harabasz(X, labels):
        X = np.asarray(X, float); labels = np.asarray(labels); ks = np.unique(labels); n, K = len(X), len(ks)
        m = X.mean(0); B = W = 0.0
        for k in ks:
            Xk = X[labels == k]; ck = Xk.mean(0)
            B += len(Xk) * ((ck - m) ** 2).sum(); W += ((Xk - ck) ** 2).sum()
        return (B / (K - 1)) / (W / (n - K)) if W > 0 else np.inf

    @staticmethod
    def davies_bouldin(X, labels):
        X = np.asarray(X, float); labels = np.asarray(labels); ks = np.unique(labels)
        cen = np.array([X[labels == k].mean(0) for k in ks])
        S = np.array([np.linalg.norm(X[labels == k] - cen[i], axis=1).mean() for i, k in enumerate(ks)])   # độ phân tán trong cụm
        M = cdist(cen, cen); np.fill_diagonal(M, np.inf)
        R = (S[:, None] + S[None, :]) / M
        return np.mean(R.max(1))

    @staticmethod
    def dunn(X, labels):
        X = np.asarray(X, float); labels = np.asarray(labels); ks = np.unique(labels)
        cum = [X[labels == k] for k in ks]
        duong_kinh = max(cdist(c, c).max() for c in cum)                       # đường kính lớn nhất trong cụm
        tach = min(cdist(cum[i], cum[j]).min() for i in range(len(ks)) for j in range(i + 1, len(ks)))   # khoảng cách giữa cụm nhỏ nhất
        return tach / duong_kinh

    @staticmethod
    def dbcv(X, labels):
        """Density-Based Clustering Validation (Moulavi và cs., 2014)."""
        X = np.asarray(X, float); labels = np.asarray(labels); n, d = X.shape
        ks = [k for k in np.unique(labels) if k != -1]
        if len(ks) < 2: return 0.0
        core, noi_bo, mrd_c, dspc = {}, {}, {}, {}
        for k in ks:
            idx = np.where(labels == k)[0]; Xk = X[idx]; nk = len(idx)
            D = cdist(Xk, Xk); np.fill_diagonal(D, np.inf)
            # all-points core distance: ( Σ (1/dist)^d / (nk-1) )^(-1/d)
            with np.errstate(divide='ignore'):
                kc = (np.where(np.isinf(D), 0, (1.0 / D) ** d).sum(1) / max(nk - 1, 1)) ** (-1.0 / d)
            np.fill_diagonal(D, 0); core[k] = kc
            MRD = np.maximum(np.maximum(kc[:, None], kc[None, :]), D); np.fill_diagonal(MRD, 0); mrd_c[k] = MRD
            T = minimum_spanning_tree(MRD + 1e-12 * (1 - np.eye(nk))).toarray(); T = np.maximum(T, T.T)
            deg = (T > 0).sum(1); inner = np.where(deg > 1)[0]
            if len(inner) == 0: inner = np.arange(nk)                           # cụm ≤ 2 điểm
            noi_bo[k] = idx[inner]
            sub = T[np.ix_(inner, inner)]
            dspc[k] = sub.max() if sub.max() > 0 else T.max()                   # độ thưa mật độ của cụm
        # core distance (của từng điểm theo cụm của nó) – dùng cho khoảng cách tách giữa cụm
        core_all = np.zeros(n)
        for k in ks: core_all[np.where(labels == k)[0]] = core[k]
        tong = 0.0; N = len(labels)
        for k in ks:
            dsc = np.inf
            for j in ks:
                if j == k: continue
                A, B = noi_bo[k], noi_bo[j]
                M = np.maximum(np.maximum(core_all[A][:, None], core_all[B][None, :]), cdist(X[A], X[B]))
                dsc = min(dsc, M.min())                                         # độ tách mật độ
            vc = (dsc - dspc[k]) / max(dsc, dspc[k])
            tong += (labels == k).sum() / N * vc
        return tong

    @classmethod
    def tat_ca(cls, X, labels):
        return {'Silhouette': cls.silhouette(X, labels), 'Calinski-Harabasz': cls.calinski_harabasz(X, labels),
                'Davies-Bouldin': cls.davies_bouldin(X, labels), 'Dunn': cls.dunn(X, labels), 'DBCV': cls.dbcv(X, labels)}

**Kiểm tra tính đúng đắn** bằng cách đối chiếu 3 độ đo có sẵn trong sklearn (Dunn và DBCV không có trong sklearn nên kiểm tra bằng bộ dữ liệu có cấu trúc rõ ràng).

In [ ]:
from sklearn.datasets import make_blobs, make_moons
Xt, yt = make_blobs(300, centers=3, cluster_std=0.8, random_state=3)
mine = Cluster_Validity.tat_ca(Xt, yt)
print('Tự cài đặt:', {k: round(v, 4) for k, v in mine.items()})
print('sklearn   : Silhouette=%.4f | CH=%.4f | DB=%.4f' % (skm.silhouette_score(Xt, yt), skm.calinski_harabasz_score(Xt, yt), skm.davies_bouldin_score(Xt, yt)))

# DBCV: cụm hình trăng đúng nhãn phải cho DBCV CAO hơn nhãn gán ngẫu nhiên; Silhouette thì ngược lại (không phù hợp cụm không lồi)
Xm, ym = make_moons(300, noise=0.06, random_state=1); rnd = np.random.default_rng(0).integers(0, 2, 300)
print('\nMoons – nhãn đúng : DBCV = %.3f | Silhouette = %.3f' % (Cluster_Validity.dbcv(Xm, ym), Cluster_Validity.silhouette(Xm, ym)))
print('Moons – nhãn ngẫu nhiên: DBCV = %.3f | Silhouette = %.3f' % (Cluster_Validity.dbcv(Xm, rnd), Cluster_Validity.silhouette(Xm, rnd)))

---
## 2. Dữ liệu Shopping.csv

In [ ]:
def load_shopping():
    p = folder + 'Shopping.csv'
    if os.path.exists(p): return pd.read_csv(p), 'file'
    rng = np.random.default_rng(5); tam = [(25, 78), (25, 20), (55, 50), (88, 17), (88, 82)]   # (income k$, spending score)
    rows = [np.c_[rng.normal(a, 6, 40), rng.normal(b, 8, 40)] for a, b in tam]
    A = np.vstack(rows)
    return pd.DataFrame({'CustomerID': np.arange(1, 201), 'Gender': rng.choice(['Male', 'Female'], 200),
                         'Age': rng.integers(18, 70, 200), 'Annual Income (k$)': A[:, 0].clip(5).round(),
                         'Spending Score (1-100)': A[:, 1].clip(1, 100).round()}), 'mô phỏng'

shop, nguon = load_shopping(); print('Nguồn:', nguon, shop.shape); display(shop.head()) if 'display' in globals() else print(shop.head())
# Chọn các cột số (bỏ ID); mặc định dùng thu nhập và điểm chi tiêu để dễ vẽ
so = [c for c in shop.select_dtypes('number').columns if 'id' not in c.lower()]
cot = [c for c in so if 'income' in c.lower() or 'spending' in c.lower()] or so[-2:]
print('Features dùng gom cụm:', cot)
X = shop[cot].values.astype(float)
Xs = StandardScaler().fit_transform(X)       # chuẩn hóa vì k-Means/FCM dựa vào khoảng cách

### 2.1 k-Means và Fuzzy C-Means tự cài đặt (tham số k cho trước)
| | k-Means | Fuzzy C-Means |
|---|---|---|
| Gán cụm | cứng (mỗi điểm 1 cụm) | **mềm**: độ thuộc uᵢⱼ ∈ [0,1], tổng theo cụm = 1 |
| Cập nhật tâm | trung bình các điểm của cụm | trung bình có trọng số uᵢⱼᵐ |
| KHI NÀO DÙNG | cụm lồi, cầu, kích thước tương đương; dữ liệu lớn | các cụm **chồng lấn/ranh giới mờ** (khách hàng thuộc nhiều nhóm), cần biết mức độ thuộc cụm |

In [ ]:
class KMeans_Scratch:
    def __init__(self, k=3, n_init=10, max_iter=300, tol=1e-6, seed=0):
        self.k, self.n_init, self.max_iter, self.tol, self.seed = k, n_init, max_iter, tol, seed

    def _kmeanspp(self, X, rng):
        c = [X[rng.integers(len(X))]]
        for _ in range(self.k - 1):
            d2 = cdist(X, np.array(c), 'sqeuclidean').min(1); c.append(X[rng.choice(len(X), p=d2 / d2.sum())])
        return np.array(c)

    def _one(self, X, rng):
        C = self._kmeanspp(X, rng)
        for it in range(self.max_iter):
            lab = cdist(X, C, 'sqeuclidean').argmin(1)
            Cn = np.array([X[lab == j].mean(0) if (lab == j).any() else X[rng.integers(len(X))] for j in range(self.k)])
            if np.linalg.norm(Cn - C) < self.tol: C = Cn; break
            C = Cn
        lab = cdist(X, C, 'sqeuclidean').argmin(1)
        return C, lab, ((X - C[lab]) ** 2).sum(), it + 1

    def fit(self, X):
        X = np.asarray(X, float); rng = np.random.default_rng(self.seed)
        best = min((self._one(X, rng) for _ in range(self.n_init)), key=lambda r: r[2])
        self.centers_, self.labels_, self.inertia_, self.n_iter_ = best; return self

    def predict(self, X): return cdist(np.asarray(X, float), self.centers_, 'sqeuclidean').argmin(1)


class FuzzyCMeans_Scratch:
    def __init__(self, k=3, m=2.0, max_iter=300, tol=1e-6, seed=0):
        self.k, self.m, self.max_iter, self.tol, self.seed = k, m, max_iter, tol, seed

    def fit(self, X):
        X = np.asarray(X, float); n = len(X); rng = np.random.default_rng(self.seed)
        U = rng.random((n, self.k)); U /= U.sum(1, keepdims=True)
        for it in range(self.max_iter):
            Um = U ** self.m
            C = (Um.T @ X) / Um.sum(0)[:, None]                                # cập nhật tâm có trọng số
            D = cdist(X, C) + 1e-12
            Un = 1.0 / ((D[:, :, None] / D[:, None, :]) ** (2 / (self.m - 1))).sum(2)   # cập nhật độ thuộc
            if np.abs(Un - U).max() < self.tol: U = Un; break
            U = Un
        self.centers_, self.U_, self.n_iter_ = C, U, it + 1
        self.labels_ = U.argmax(1)                                              # khử mờ: cụm có độ thuộc cao nhất
        self.objective_ = ((U ** self.m) * cdist(X, C, 'sqeuclidean')).sum()
        self.pc_ = (U ** 2).sum() / n                                           # Partition Coefficient ∈ [1/k, 1]
        return self

    def predict_proba(self, X):
        D = cdist(np.asarray(X, float), self.centers_) + 1e-12
        return 1.0 / ((D[:, :, None] / D[:, None, :]) ** (2 / (self.m - 1))).sum(2)

km = KMeans_Scratch(k=5).fit(Xs); fcm = FuzzyCMeans_Scratch(k=5).fit(Xs)
print('k-Means: số vòng lặp =', km.n_iter_, '| inertia = %.3f' % km.inertia_)
print('FCM    : số vòng lặp =', fcm.n_iter_, '| objective = %.3f | PC = %.3f' % (fcm.objective_, fcm.pc_))
print('Độ thuộc của 3 khách hàng đầu (FCM):'); print(np.round(fcm.U_[:3], 3))

### 2.2 Đánh giá bằng Silhouette, Calinski-Harabasz, Davies-Bouldin, Dunn
Quét k = 2…10 để chọn số cụm. **Quy tắc chọn k:** Silhouette, CH, Dunn cực đại; Davies-Bouldin cực tiểu; ưu tiên k mà **nhiều độ đo cùng đồng ý**.

In [ ]:
class Clustering_Experiment:
    def __init__(self, X, ks=range(2, 11)): self.X, self.ks = np.asarray(X, float), list(ks)
    def quet_k(self):
        rows = []
        for k in self.ks:
            for ten, mod in [('k-Means', KMeans_Scratch(k)), ('Fuzzy C-Means', FuzzyCMeans_Scratch(k))]:
                lab = mod.fit(self.X).labels_
                if len(np.unique(lab)) < 2: continue
                m = Cluster_Validity.tat_ca(self.X, lab); rows.append({'Thuật toán': ten, 'k': k, **{x: m[x] for x in ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin', 'Dunn']}})
        self.kq = pd.DataFrame(rows); return self.kq
    def ve(self):
        cac = ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin', 'Dunn']
        fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
        for a, c in zip(ax, cac):
            for ten, g in self.kq.groupby('Thuật toán'): a.plot(g.k, g[c], 'o-', label=ten)
            a.set_title(c + (' (nhỏ tốt)' if c == 'Davies-Bouldin' else ' (lớn tốt)')); a.set_xlabel('k')
        ax[0].legend(); plt.tight_layout(); plt.show()
    def k_tot_nhat(self):
        g = self.kq.copy()
        g['Davies-Bouldin'] = -g['Davies-Bouldin']                           # đảo dấu để "lớn là tốt" cho cả 4 độ đo
        return {ten: {c: int(sub.loc[sub[c].idxmax(), 'k']) for c in ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin', 'Dunn']} for ten, sub in g.groupby('Thuật toán')}

ex = Clustering_Experiment(Xs); bang = ex.quet_k(); print(bang.round(3).to_string(index=False)); ex.ve()
print('k tốt nhất theo từng độ đo:', ex.k_tot_nhat())

**Cách đọc:** nếu các độ đo cùng chỉ về một k (với dữ liệu Mall Customers thường là k = 5) thì đó là số nhóm khách hàng hợp lý. Nếu bất đồng, ưu tiên Silhouette + Davies-Bouldin và kiểm tra thêm ý nghĩa nghiệp vụ của các cụm. Dunn rất nhạy với outlier nên dao động nhiều hơn.

In [ ]:
## Kết quả cuối với k chọn: so sánh k-Means và FCM và so với sklearn (đối chiếu)
k_chon = int(bang[bang['Thuật toán'] == 'k-Means'].sort_values('Silhouette').iloc[-1]['k']); print('k được chọn =', k_chon)
km = KMeans_Scratch(k_chon).fit(Xs); fcm = FuzzyCMeans_Scratch(k_chon).fit(Xs)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, (ten, mod) in zip(ax, [('k-Means', km), ('Fuzzy C-Means', fcm)]):
    a.scatter(X[:, 0], X[:, 1], c=mod.labels_, cmap='tab10', s=18)
    cen = StandardScaler().fit(X).inverse_transform(mod.centers_); a.scatter(cen[:, 0], cen[:, 1], c='k', marker='X', s=160)
    a.set_title(f'{ten} (k={k_chon})'); a.set_xlabel(cot[0]); a.set_ylabel(cot[1])
plt.tight_layout(); plt.show()
so_sanh = pd.DataFrame({'k-Means': Cluster_Validity.tat_ca(Xs, km.labels_), 'Fuzzy C-Means': Cluster_Validity.tat_ca(Xs, fcm.labels_)}).round(4)
print(so_sanh); print('Đối chiếu sklearn (k-Means): Silhouette=%.4f | CH=%.4f | DB=%.4f' % (skm.silhouette_score(Xs, km.labels_), skm.calinski_harabasz_score(Xs, km.labels_), skm.davies_bouldin_score(Xs, km.labels_)))

**Nhận xét:** kết quả k-Means và FCM thường rất gần nhau khi các cụm tách rõ; khác biệt nằm ở các điểm ở ranh giới – FCM cho biết độ thuộc từng cụm (ví dụ 0.55/0.45) thay vì ép vào một cụm. Với ranh giới chồng lấn, FCM cho thông tin phong phú hơn; với cụm tách rõ và dữ liệu lớn, k-Means nhanh và đủ dùng.

---
## 3. Lập trình hướng đối tượng
Các lớp: `Cluster_Validity` (5 độ đo), `KMeans_Scratch`, `FuzzyCMeans_Scratch`, `Clustering_Experiment` (quét k, vẽ, chọn k). Cùng giao diện `fit(X)` → `labels_`, `centers_` nên có thể thay thế thuật toán khác (DBSCAN…) và đánh giá bằng cùng bộ độ đo.